# Making It Fast: Quantization, Speculative Decoding, KV Optimization**Section 3 | 20 minutes**Three levers to double throughput without changing your model architecture or hardware.

In [ ]:
import numpy as npimport matplotlib.pyplot as pltplt.style.use('seaborn-v0_8-whitegrid')params_B = 7  # Llama-2 7B as reference

## Lever 1: QuantizationFewer bytes per weight = faster memory reads = higher throughput.GPU compute is rarely the bottleneck during decode. Memory bandwidth is. Quantization directly attacks this.

In [ ]:
precisions = ['FP16', 'INT8', 'INT4']bytes_per_param = [2, 1, 0.5]model_sizes_gb = [params_B * b for b in bytes_per_param]# A100 80GB: 2 TB/s bandwidthbw_tb_s = 2.0# Throughput ceiling (tokens/s) = bandwidth / (model_size * bytes_read_per_token)# Simplified: max decode tokens/s ≈ bandwidth / model_sizethroughput_ceiling = [bw_tb_s * 1024 / sz for sz in model_sizes_gb]fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))colors = ['#2563eb', '#16a34a', '#dc2626']ax1.bar(precisions, model_sizes_gb, color=colors, edgecolor='black')ax1.set_ylabel('Model Size (GB)')ax1.set_title(f'Llama-2 {params_B}B: Size by Precision')for i, v in enumerate(model_sizes_gb): ax1.text(i, v + 0.2, f'{v:.1f} GB', ha='center', fontweight='bold')ax2.bar(precisions, throughput_ceiling, color=colors, edgecolor='black')ax2.set_ylabel('Max Decode Tokens/s')ax2.set_title('Throughput Ceiling (A100 80GB)')for i, v in enumerate(throughput_ceiling): ax2.text(i, v + 5, f'{v:.0f}', ha='center', fontweight='bold')plt.tight_layout()plt.show()print(f"INT4 gives {throughput_ceiling[2]/throughput_ceiling[0]:.1f}x throughput ceiling vs FP16")

## The Quality TradeoffQuantization isn't free. Lower precision introduces approximation error. The key insight from research: INT4 with proper calibration (GPTQ, AWQ) loses surprisingly little quality.

In [ ]:
# Data from published benchmarks (GPTQ, AWQ, QuIP# papers)data = {    'Precision': ['FP16 (baseline)', 'INT8 (absmax)', 'INT4 (GPTQ)', 'INT4 (AWQ)', 'INT4 (QuIP#)'],    'Perplexity (Wiki)': [5.68, 5.70, 5.85, 5.78, 5.74],    'MMLU (%)': [45.3, 45.2, 44.8, 45.0, 45.1],    'Size (GB)': [14.0, 7.0, 3.5, 3.5, 3.5],    'Δ Perplexity': [0.0, 0.02, 0.17, 0.10, 0.06],}print(f"{'Precision':<18} {'Perplexity':<14} {'MMLU':<10} {'Size':<10} {'Δ PPL'}")print("-" * 62)for i in range(len(data['Precision'])):    print(f"{data['Precision'][i]:<18} {data['Perplexity (Wiki)'][i]:<14.2f} {data['MMLU (%)'][i]:<10.1f} {data['Size (GB)'][i]:<10.1f} {data['Δ Perplexity'][i]:+.2f}")print("\n→ AWQ/QuIP# achieve 4x compression with <0.1 perplexity increase")

## Lever 2: Speculative DecodingUse a small draft model to propose γ tokens, then verify all at once with the large model in a single forward pass. Accepted tokens are "free" (parallel verification costs ~1 forward pass regardless of γ).

In [ ]:
# Speculative decoding speedup formula:# Speedup ≈ (1 - α^(γ+1)) / (1 - α) / cost_ratio# where α = acceptance rate, γ = draft length, cost_ratio ≈ 1 + γ*c (c = draft/target cost)gammas = [3, 5, 7, 10]acceptance_rates = np.linspace(0.5, 0.95, 50)draft_cost_ratio = 0.05  # draft model is ~5% cost of targetfig, ax = plt.subplots(figsize=(8, 5))colors = ['#2563eb', '#16a34a', '#f59e0b', '#dc2626']for gamma, color in zip(gammas, colors):    speedups = []    for alpha in acceptance_rates:        expected_tokens = (1 - alpha**(gamma + 1)) / (1 - alpha)        cost = 1 + gamma * draft_cost_ratio  # one target pass + γ draft passes        speedup = expected_tokens / cost        speedups.append(speedup)    ax.plot(acceptance_rates, speedups, label=f'γ={gamma}', color=color, linewidth=2)ax.axhline(y=1, color='gray', linestyle='--', alpha=0.5, label='No speedup')ax.set_xlabel('Acceptance Rate (α)')ax.set_ylabel('Speedup vs Autoregressive')ax.set_title('Speculative Decoding: Speedup vs Acceptance Rate')ax.legend()ax.set_ylim(0.5, 6)plt.tight_layout()plt.show()print("Key insight: γ=5 with 80% acceptance → ~3x speedup")

## Lever 3: KV Cache EngineeringEvery token stored in the KV cache costs memory. Multi-Head Attention (MHA) stores one KV pair per head. Grouped-Query Attention (GQA) and Multi-Latent Attention (MLA) dramatically reduce this.

In [ ]:
# KV cache memory per token (Llama-2 7B scale: 32 layers, d_model=4096)n_layers = 32d_model = 4096n_heads = 32d_head = d_model // n_heads  # 128# Memory per token = 2 (K+V) * n_layers * n_kv_heads * d_head * bytesconfigs = {    'MHA (32 KV heads)': 32,    'GQA-8 (8 KV heads)': 8,    'GQA-4 (4 KV heads)': 4,    'MLA (compressed)': 1,  # ~equivalent to 1 head via low-rank projection}bytes_fp16 = 2mem_per_token = {}for name, kv_heads in configs.items():    mem = 2 * n_layers * kv_heads * d_head * bytes_fp16  # bytes per token    mem_per_token[name] = mem / 1024  # KBnames = list(mem_per_token.keys())values = list(mem_per_token.values())colors = ['#dc2626', '#f59e0b', '#16a34a', '#2563eb']fig, ax = plt.subplots(figsize=(8, 4))bars = ax.barh(names, values, color=colors, edgecolor='black')ax.set_xlabel('KV Cache per Token (KB)')ax.set_title('KV Cache Memory: Architecture Comparison (32 layers, d=4096)')for bar, v in zip(bars, values):    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2, f'{v:.0f} KB', va='center', fontweight='bold')plt.tight_layout()plt.show()savings = values[0] / values[1]print(f"GQA-8 saves {savings:.0f}x memory vs MHA → {savings:.0f}x more tokens in same GPU memory")print(f"MLA saves {values[0]/values[-1]:.0f}x → enables 100K+ context lengths")

## Combined EffectThese optimizations compose multiplicatively. Let's compute the total improvement when stacking all three.

In [ ]:
# Combined optimization stackquant_speedup = 4.0       # INT4 vs FP16: 4x less memory bandwidthspec_speedup = 2.5        # Speculative (γ=5, α=0.75): ~2.5xkv_savings = 4.0          # GQA-8: 4x less KV memory → 4x longer context or 4x more concurrent users# Throughput improvement (quant * speculative)throughput_factor = quant_speedup * spec_speedup# Memory efficiency (quant model + GQA cache)# Model: 4x smaller. KV cache: 4x smaller per token.effective_capacity = quant_speedup * kv_savings  # concurrent users at same memory budgetprint("=" * 50)print("OPTIMIZATION STACK: INT4 + Speculative + GQA-8")print("=" * 50)print(f"\n{'Quantization (INT4):':<30} {quant_speedup:.0f}x throughput ceiling")print(f"{'Speculative decoding:':<30} {spec_speedup:.1f}x decode speedup")print(f"{'GQA-8 KV cache:':<30} {kv_savings:.0f}x memory per token")print(f"\n{'Combined throughput:':<30} {throughput_factor:.0f}x faster generation")print(f"{'Combined capacity:':<30} {effective_capacity:.0f}x more concurrent users")print(f"\n→ From ~146 tok/s to ~{146*throughput_factor:.0f} tok/s on A100")print(f"→ From serving 50 users to {50*int(effective_capacity)} users on same hardware")print("\nAll achieved WITHOUT changing the base model or buying more GPUs.")